# 服の 3D の試作: StdGEN(ステップ 6b)

服を着た素体の絵(正面1枚、背景を透明にしたもの)から、StdGEN(CVPR 2025)で **体・服・髪を分けた 3D** を作る。
もう一つの方法(着せた正面・横・後ろの絵から Hunyuan3D-2mv で作り、体との差で服を切り出す)は、手元とアプリの Colab
サーバーで作るので、このノートブックでは動かさない。計画は docs/model_3d_plan_jp.md のステップ 6b。

- StdGEN のコードは Apache-2.0、学習済みの重みは「研究用」。この試作は、利用者自身の Colab で、利用者の絵だけで動かす
  (公開の Space には絵を送らない)。
- StdGEN は Python 3.9・torch 2.1(CUDA 11.8)を前提にしているので、Colab の Python とは別に uv で 3.9 の環境を作る。

1. ランタイムを **GPU(A100 推奨。L4 でも `LOW_VRAM = True` で動く見込み)** にする。
2. 上から順に実行する。セル「入力」で `stdgen_inputs.zip` を選んでアップロードする。
3. 最後のセルの `stdgen_results.zip` をダウンロードして、アプリの作業者に渡す。

結果は `/content/StdGEN/result/`(A ポーズ・多視点・粗い形・仕上げ)と、時間・VRAM の記録 `/content/stdgen_log.json`。

In [ ]:
# GPU の確認
!nvidia-smi --query-gpu=name,memory.total --format=csv
LOW_VRAM = False  # L4(24GB)で足りなければ True(多視点の段で、モデルを CPU に逃がす。遅くなる)

## 準備(30〜40 分。pytorch3d のビルドに時間がかかる)

In [ ]:
%%bash
set -e
cd /content
pip -q install uv
[ -d StdGEN ] || git clone -q https://github.com/hyz317/StdGEN.git
cd StdGEN
# StdGEN 用の Python 3.9 の環境(Colab の Python とは別)。Colab は uv に「システムの Python に入れる」設定を
# していることがあるので、入れる先の Python を毎回はっきり指定する。前に失敗したときの環境は作り直す
unset UV_SYSTEM_PYTHON
uv python install 3.9
rm -rf /content/venv39
uv venv -q --seed -p 3.9 /content/venv39
PY=/content/venv39/bin/python
$PY --version
# Colab の CUDA は 13 になったので、torch 2.1(CUDA 12.1 版)に合う CUDA 12 の nvcc を別に入れる(拡張のビルド用)
. /etc/os-release
REPO=ubuntu${VERSION_ID/./}
if [ ! -d /usr/local/cuda-12 ] && ! ls -d /usr/local/cuda-12.* >/dev/null 2>&1; then
  wget -q https://developer.download.nvidia.com/compute/cuda/repos/$REPO/x86_64/cuda-keyring_1.1-1_all.deb
  dpkg -i cuda-keyring_1.1-1_all.deb >/dev/null
  apt-get -qq update
  for v in 12-6 12-5 12-4 12-1; do
    if apt-cache show cuda-nvcc-$v >/dev/null 2>&1; then
      apt-get -qq install -y cuda-nvcc-$v cuda-cudart-dev-$v cuda-libraries-dev-$v cuda-nvrtc-dev-$v >/dev/null
      break
    fi
  done
fi
export CUDA_HOME=$(ls -d /usr/local/cuda-12.* | sort -V | tail -1)
export PATH=$CUDA_HOME/bin:$PATH
echo "CUDA_HOME=$CUDA_HOME" > /content/stdgen_cuda.env
nvcc --version | tail -1
uv pip install -q --python $PY torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu121
uv pip install -q --python $PY xformers==0.0.22.post7 --index-url https://download.pytorch.org/whl/cu121
# torch-scatter は出来合いの wheel を直に入れる(探す場所を渡すだけだと、ソースからビルドしようとして失敗した)
uv pip install -q --python $PY "https://data.pyg.org/whl/torch-2.1.0%2Bcu121/torch_scatter-2.1.2%2Bpt21cu121-cp39-cp39-linux_x86_64.whl"
# requirements.txt のうち、ビルドに torch の要るもの(nvdiffrast・pytorch3d)は、別の環境を作らずにビルドする
grep -v -E "nvdiffrast|pytorch3d" requirements.txt > /tmp/stdgen_req.txt
uv pip install -q --python $PY -r /tmp/stdgen_req.txt
# torch 2.1 の拡張のビルド(cpp_extension)は pkg_resources.packaging を使うので、それを外した setuptools 70 より前にする
uv pip install -q --python $PY "setuptools<70" wheel ninja
export TORCH_CUDA_ARCH_LIST="8.0;8.6;8.9" FORCE_CUDA=1
uv pip install -q --python $PY --no-build-isolation git+https://github.com/NVlabs/nvdiffrast
# pytorch3d は Python 3.9・torch 2.1・CUDA 12.1 の出来合いが無いので、ここでビルドする(15〜25 分)
uv pip install --python $PY --no-build-isolation "git+https://github.com/facebookresearch/pytorch3d.git@stable"
$PY -c "import torch, pytorch3d, xformers, nvdiffrast.torch; print('torch', torch.__version__, torch.cuda.is_available(), 'pytorch3d', pytorch3d.__version__)"


In [ ]:
%%bash
set -e
cd /content/StdGEN
PY=/content/venv39/bin/python
mkdir -p ckpt
# 学習済みの重み(研究用)と、SAM(ViT-H)
$PY -c "from huggingface_hub import snapshot_download; snapshot_download('hyz317/StdGEN', local_dir='ckpt')"
[ -f ckpt/sam_vit_h_4b8939.pth ] || wget -q -O ckpt/sam_vit_h_4b8939.pth https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
ls ckpt


## 入力

`stdgen_inputs.zip`(中に `<名前>.png`。背景を透明にした、服を着た素体の正面の絵)を選ぶ。

In [ ]:
import zipfile, pathlib
from google.colab import files
up = files.upload()
inp = pathlib.Path('/content/StdGEN/input_cases'); inp.mkdir(parents=True, exist_ok=True)
for name in up:
    with zipfile.ZipFile(name) as z:
        z.extractall(inp)
print(sorted(p.name for p in inp.iterdir()))

## 実行(段ごとに時間と GPU メモリを記録する。失敗した段は記録に残して止める)

In [ ]:
import json, os, subprocess, time, pathlib
# 準備で入れた CUDA 12(nvdiffrast が動くときにもビルドすることがある)
cuda = pathlib.Path('/content/stdgen_cuda.env').read_text().strip().split('=', 1)[1]
env = {**os.environ, 'CUDA_HOME': cuda, 'PATH': f"{cuda}/bin:" + os.environ['PATH']}
# Colab の matplotlib の設定(ノートブックの中に描く)は、3.9 の環境の matplotlib では使えない
env['MPLBACKEND'] = 'Agg'
py = '/content/venv39/bin/python'
steps = [
    ('apose', ['infer_canonicalize.py', '--input_dir', './input_cases', '--output_dir', './result/apose']),
    ('multiview', ['infer_multiview.py', '--input_dir', './result/apose', '--output_dir', './result/multiview'] + (['--low_vram'] if LOW_VRAM else [])),
    ('slrm', ['infer_slrm.py', '--input_dir', './result/multiview', '--output_dir', './result/slrm']),
    ('refine', ['infer_refine.py', '--input_mv_dir', 'result/multiview', '--input_obj_dir', './result/slrm', '--output_dir', './result/refine']),
]
log = {}
for name, args in steps:
    started = time.time()
    watch = subprocess.Popen(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits', '-lms', '500'],
                             stdout=subprocess.PIPE, text=True)
    run = subprocess.run([py, *args], cwd='/content/StdGEN', capture_output=True, text=True, env=env)
    watch.terminate()
    peak = max((int(x) for x in watch.stdout.read().split() if x.isdigit()), default=0)
    log[name] = {'seconds': round(time.time() - started), 'peak_mib': peak, 'ok': run.returncode == 0,
                 'tail': (run.stdout + run.stderr)[-3000:]}
    print(name, log[name]['seconds'], 's', peak, 'MiB', 'ok' if run.returncode == 0 else 'NG')
    if run.returncode != 0:
        print(log[name]['tail'])
        break
pathlib.Path('/content/stdgen_log.json').write_text(json.dumps(log, ensure_ascii=False, indent=1))

In [ ]:
# できたファイルの一覧(体・服・髪が別のファイルか、を見る)
!find /content/StdGEN/result -maxdepth 3 -type f | sort | head -80

## 結果をまとめてダウンロード

In [ ]:
!cd /content && rm -f stdgen_results.zip && zip -qr stdgen_results.zip StdGEN/result stdgen_log.json && ls -lh stdgen_results.zip
from google.colab import files
files.download('/content/stdgen_results.zip')